# Lab 2_1B Companion: Changing, Building, and Flashing `basic-passwdcheck`

This notebook is a **standalone, self-contained walkthrough** of one specific task that
comes up while working through *Lab 2_1B - Power Analysis for Password Bypass*: changing
the password baked into the target firmware, rebuilding it, and getting the new build
onto the real ChipWhisperer Husky board.

It exists because doing this for the first time hits a handful of non-obvious errors -
this notebook writes down exactly what those errors looked like, why they happened, and
what fixed them, so the next person doesn't have to rediscover it.

**Who this is for:** anyone who wasn't part of the original troubleshooting session and
just wants to know: *which file do I edit, how do I rebuild it, and how do I get it onto
the board.* No prior context is assumed beyond having the hardware from Lab 0 connected.

**Relationship to the other 2_1B notebooks:**
* `Lab 2_1B - Power Analysis for Password Bypass (MAIN).ipynb` is the actual lab - it runs
  the SPA attack that recovers the password and (if you have hardware) lets you try
  guesses interactively. Its **"Optional: Attack Your Own Password"** section briefly
  covers editing/rebuilding/reflashing.
* *This* notebook expands that into a full walkthrough with a working, tested flashing
  script and a troubleshooting log of every error hit while getting it running - useful
  if you want to flash the board directly (e.g. from a plain terminal) rather than through
  the lab notebook's own setup cells.

## Prerequisites

* **Hardware:** ChipWhisperer Husky (capture board) + CW313 baseboard + CW312T-SAM4S
  target, assembled and connected over USB as described in **Lab 0**. The target must be
  the **SAM4S** board, not the iCE40 FPGA target that sometimes ships in the same kit box
  - the SAM4S bootloader is what the flashing step below talks to.
* **Software environment:** a Python environment with `chipwhisperer` installed (this repo
  is normally used through the `cwenv` virtual environment), plus an ARM toolchain
  (`arm-none-eabi-gcc`) and `make` on your `PATH`. On Windows, use the "CW Compiler
  Environment" shell if you have it, or make sure Git's `usr/bin` (which provides `rm`,
  `sh`) is reachable - the build cell below works around this automatically either way.
* **Paths below** are relative to this notebook's own location,
  `jupyter/courses/sca101/`, and point into the sibling `firmware` checkout at
  `../../../firmware/mcu/basic-passwdcheck/`.

## Files involved

| File | What it is |
|---|---|
| `../../../firmware/mcu/basic-passwdcheck/basic-passwdcheck.c` | The firmware source. **This is the only file you need to edit** - `correct_passwd[]` near line 73 is the password the target compares your input against. |
| `../../../firmware/mcu/basic-passwdcheck/makefile` | The build recipe for this one firmware project (pulls in the shared `Makefile.simpleserial` / `Makefile.inc`). You don't need to edit this - just know it needs a `PLATFORM=` argument (see Step 2). |
| `../../../firmware/mcu/basic-passwdcheck/basic-passwdcheck-CWHUSKY.hex` (and `.bin`, `.elf`, `.map`, ...) | Build **output**. Regenerated every time you run `make`; this is the file that actually gets written to the target's flash. |
| `../../../firmware/mcu/basic-passwdcheck/flash_husky.py` | A small standalone Python script (written and validated in this session) that connects to the Husky and flashes the `.hex` above. Lets you reflash from a plain terminal without opening any lab notebook. |

Everything else in that directory (`objdir-CWHUSKY/`, `.lst`/`.o`/`.sym`/`.lss` files) is
intermediate build output - safe to ignore, and cleaned up by `make clean`.

---
## Step 1 - Change the password in the firmware source

The password is a plain C string literal, hardcoded in `main()`:

```c
char passwd[32];
char correct_passwd[] = "h0px3";
```

Open `basic-passwdcheck.c` in a text editor and change the string on the
`correct_passwd[]` line to whatever you want, for example:

```c
char correct_passwd[] = "lfbc1";
```

That's it - no other line needs to change to make the firmware itself work. At runtime
this string is copied onto the stack and compared one byte at a time against whatever you
type at the "Please enter password" prompt; the comparison loop `break`s on the first
mismatched byte, which is exactly the timing/power leak the SPA attack in the MAIN
notebook exploits.

**If you also want the automated attack in the MAIN notebook to find your new password:**
it only sweeps the character set `abcdefghijklmnopqrstuvwxyz0123456789`, and its loop is
hardcoded to try exactly 5 characters. Keep your password to 5 characters from that set,
or update the attack loop's `range(0, N)`, the final `assert`, and `EXPECTED_PW` to match
a different length - see "Optional: Attack Your Own Password" in the MAIN notebook for the
full explanation of those constraints. None of that is required just to build and flash
the firmware, only to have the existing attack code recover it automatically.

The cell below doesn't change anything - it just reads the file back so you can confirm
you edited the right line before moving on:

In [ ]:
import pathlib

fw_c = pathlib.Path('../../../firmware/mcu/basic-passwdcheck/basic-passwdcheck.c')
for lineno, line in enumerate(fw_c.read_text().splitlines(), start=1):
    if 'correct_passwd' in line:
        print(f"{lineno}: {line}")

---
## Step 2 - Build the firmware

The build is a normal `make`-based AVR/ARM firmware build, but this project's `makefile`
supports many different target boards and **won't guess which one you want**. Running it
with no arguments:

```bash
cd firmware/mcu/basic-passwdcheck
make
```

fails with a long table listing every supported `PLATFORM` (CW304, CW308_STM32F4, CWHUSKY,
...) followed by:

```
Options to define platform:
(1) Run make with PLATFORM specified as follows:
make PLATFORM=CW304
...
.  Stop.
```

This is expected, not a real error - it's `make` telling you it needs a `PLATFORM=` value.
For the Husky + CW312T-SAM4S kit, the platform name is **`CWHUSKY`** (it's an alias for
`CW312_SAM4S` internally). The correct command is:

```bash
cd firmware/mcu/basic-passwdcheck
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1 -j
```

* `PLATFORM=CWHUSKY` - selects the Husky/SAM4S target (do **not** use the older
  `CW308_SAM4S` string; it's deprecated and matches a different, unmaintained config).
* `CRYPTO_TARGET=NONE` - this firmware doesn't do any AES/crypto, so no crypto backend
  needs to be compiled in.
* `SS_VER=SS_VER_2_1` - selects version 2.1 of the SimpleSerial communication protocol.
  This has to match whatever the Python side (`cw.target(scope, cw.targets.SimpleSerial2)`
  or the MAIN notebook's `SS_VER` setting) expects, or the two sides won't understand each
  other over serial even though flashing succeeds.
* `-j` - build in parallel; purely a speed-up.

Add `clean` before `all` (or just `make clean` first) if you want to force a full rebuild
instead of an incremental one - useful if you've changed platforms or something looks
stale, but not required for a simple string-literal edit like this.

The cell below runs that exact command via `subprocess` so this notebook is runnable
top-to-bottom. On Windows it also works around one build-environment gotcha: the
makefile's clean step shells out to `rm`, which isn't always on `PATH` outside of a
Unix-like shell - the helper below finds Git's bundled `usr/bin` (which ships `rm`) and
adds it for the build if needed.

In [ ]:
import os
import shutil
import subprocess
import pathlib

def _build_env():
    """make's clean step needs Unix coreutils (rm, sh, mkdir). On Windows, add
    Git's usr/bin to PATH if they aren't already there."""
    env = os.environ.copy()
    if os.name == "nt" and not shutil.which("rm"):
        base = shutil.which("git") or ""
        for _ in range(4):
            base = os.path.dirname(base)
            cand = os.path.join(base, "usr", "bin")
            if os.path.isfile(os.path.join(cand, "rm.exe")):
                env["PATH"] = cand + os.pathsep + env["PATH"]
                return env
        print("WARNING: 'rm' not found on PATH. If the build fails below, run this "
              "from the 'CW Compiler Environment' shell instead.")
    return env

FW_DIR = pathlib.Path('../../../firmware/mcu/basic-passwdcheck')

result = subprocess.run(
    ['make', 'PLATFORM=CWHUSKY', 'CRYPTO_TARGET=NONE', 'SS_VER=SS_VER_2_1', '-j'],
    cwd=FW_DIR, capture_output=True, text=True, env=_build_env(),
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError('firmware build failed - see output above')

hex_file = FW_DIR / 'basic-passwdcheck-CWHUSKY.hex'
print(f"\nBuild finished (exit code {result.returncode}).")
print(f"Output: {hex_file.resolve()} ({hex_file.stat().st_size} bytes)")

---
## Step 3 - Flash the compiled firmware to the Husky

There are two ways to do this, depending on where you are:

**A) From inside a lab notebook** that has already run `Setup_Generic.ipynb` (which is
what `Lab 2_1B ... (HARDWARE).ipynb` does) - `scope` and `target` already exist, fully
configured, and flashing is a one-liner:

```python
cw.program_target(scope, cw.programmers.SAM4SProgrammer, hex_path)
```

**B) Standalone**, e.g. from a plain terminal with no lab notebook open at all - which is
what the rest of this section builds. This is useful for quickly testing a firmware change
without spinning up the whole lab notebook's setup.

### Connecting and flashing manually

Doing this outside of `Setup_Generic.ipynb`'s helper code means recreating the couple of
setup steps it normally does for you. The cell below is the exact, working sequence -
each line is explained after a warning we hit that pointed straight at it:

```python
scope = cw.scope()
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
cw.program_target(scope, cw.programmers.SAM4SProgrammer, hex_path)
```

* `cw.scope()` opens the connection to the Husky capture board itself.
* `scope.io.tio1` / `tio2` wire two of the Husky's general-purpose IO lines to the
  target's UART (receive/transmit). The SAM4S's bootloader is programmed *over that
  serial link*, so without this the programmer can't talk to the chip at all. Leaving
  this out produces exactly this warning, followed by a timeout:
  ```
  Serial pins incorrect for NewAE SAM4S target
  (tio1, tio2) != ('serial_rx', 'serial_tx')
  Did you forget to call scope.default_setup()?
  ```
  Normally `scope.default_setup()` sets these two lines for you (along with a lot of
  other setup unrelated to flashing). We set them directly instead here - see the
  Troubleshooting section below for why.
* `cw.program_target(scope, ProgrammerClass, path)` is the actual flash step. Pass the
  **class**, not an instance - `cw.program_target` constructs it internally.
  `SAM4SProgrammer` speaks Atmel's SAM-BA bootloader protocol, which is what the SAM4S
  chip on the CW312T target uses.

Run the cell below to flash the board now (skip it if you just want to read the
walkthrough):

In [ ]:
import chipwhisperer as cw

hex_path = str((FW_DIR / 'basic-passwdcheck-CWHUSKY.hex').resolve())

scope = cw.scope()
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
print("Connected to Husky.")

cw.program_target(scope, cw.programmers.SAM4SProgrammer, hex_path)
print("Flashed:", hex_path)

### Doing the same thing from a terminal, without Jupyter

The exact same three-step sequence is saved as a standalone script,
`../../../firmware/mcu/basic-passwdcheck/flash_husky.py`, so you can rebuild and reflash
without opening Jupyter at all:

```bash
cd firmware/mcu/basic-passwdcheck
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1 -j
python flash_husky.py
```

A couple of Windows/PowerShell-specific wrinkles came up running it this way - both are
covered in the Troubleshooting section right below (path quoting, and stray processes
holding the USB connection open).

**Confirmed working end-to-end:** `flash_husky.py` was run against a real Husky +
CW312T-SAM4S and completed successfully:
```
Firmware path: ...asic-passwdcheck-CWHUSKY.hex
Firmware size: 7452 bytes

Connecting to ChipWhisperer Husky...
Connected to Husky

Programming firmware...
Firmware flashed successfully!

Done. Your Husky is now running the modified basic-passwdcheck firmware.
```

---
## Troubleshooting log

Every error actually hit while getting this working, in the order they showed up, with
the fix:

| # | Symptom | Cause | Fix |
|---|---|---|---|
| 1 | `make` prints a large table of every supported `PLATFORM` and stops with `. Stop.` | Ran `make` (or `make clean all`) with no `PLATFORM=` argument - the makefile supports dozens of boards and refuses to guess. | Always pass it explicitly: `make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1 -j`. |
| 2 | PowerShell: `python.exe: can't open file '...\flash_husky.py\': [Errno 22] Invalid argument` | A trailing backslash got appended to the path (e.g. from tab-completion/copy-paste), and PowerShell's escaping of a quoted Windows path with a trailing `\` broke the argument. | Run it with a plain relative path from inside the directory: `python flash_husky.py` (or `python .\flash_husky.py`), or use forward slashes in a full path. |
| 3 | `Error connecting to scope: Unable to communicate with found ChipWhisperer. Check that another process isn't connected to it...` | Another process (very often a leftover Jupyter kernel, or a previous script run that never called `scope.dis()`) still has the USB device open. | Check for stray processes (`Get-Process python` in PowerShell) and close them / shut down other Jupyter kernels; unplug and replug the Husky if it persists. |
| 4 | `AttributeError: 'NoneType' object has no attribute 'clock'`, raised from deep inside `chipwhisperer`'s Husky code (`sampling_clock_frequency` / `scope.trace.clock`) - triggered either by printing the `scope` object or by calling `scope.default_setup()` | The Husky's trace/SAD submodule (`scope.trace`) didn't come up on that particular connection, so anything that reads its clock blows up on `None`. Seen after a run of earlier failed/interrupted connection attempts. | Avoid printing the raw `scope` object, and skip `scope.default_setup()` - set only the two IO lines flashing actually needs (`scope.io.tio1`/`tio2`, see Step 3) instead of the full setup routine. If it recurs, fully power-cycle the Husky (unplug, wait, replug). |
| 5 | `'OpenADC' object has no attribute 'target'` | Wrote `scope.target.programming_flash(...)` - programming isn't a method on the scope object. | Use the module-level helper instead: `cw.program_target(scope, cw.programmers.SAM4SProgrammer, hex_path)`. |
| 6 | `Serial pins incorrect for NewAE SAM4S target` warning, followed by `✗ Error programming firmware: timeout` | The two IO lines the SAM4S programmer needs (`tio1`, `tio2`) weren't wired to the target's serial RX/TX - normally `scope.default_setup()` does this, but that was being skipped because of issue 4. | Set them explicitly before programming: `scope.io.tio1 = "serial_rx"`; `scope.io.tio2 = "serial_tx"`. |
| 7 | (Checked as a hypothesis) ~19 stray `python.exe` processes running, some hundreds of MB in size | Earlier script runs that hit an error and called `exit(1)` never reached `scope.dis()`, and/or Jupyter kernels were left running. | `Get-Process python` to list them, `Stop-Process` (or close the Jupyter tabs) to free the USB device. Worth checking early even if - as it turned out here - it isn't always the actual cause. |
| 8 | `UnicodeEncodeError: 'charmap' codec can't encode character '\u2713'` right after programming actually succeeded | The script printed a Unicode checkmark (`✓`)/cross (`✗`); Windows consoles often default to the `cp1252` codepage, which can't encode them, so the *success* message itself crashed the script. | Stick to plain ASCII in `print()` output on Windows (or set `PYTHONIOENCODING=utf-8`) - `flash_husky.py` now prints plain "Firmware flashed successfully!" instead. |

**If flashing still times out after all of the above:** double check the physical
target - it needs to be the **SAM4S** board seated on the CW313, not the iCE40 FPGA
target that sometimes ships in the same kit box. The SAM4S's bootloader is what
`SAM4SProgrammer` talks to; nothing will respond if a different chip is mounted.

---
## Recap

1. Edit `correct_passwd[]` in
   `../../../firmware/mcu/basic-passwdcheck/basic-passwdcheck.c` (the only file you need
   to change).
2. Build: `make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1 -j` from
   `firmware/mcu/basic-passwdcheck/`.
3. Flash: `cw.scope()` -> set `scope.io.tio1`/`tio2` -> `cw.program_target(scope,
   cw.programmers.SAM4SProgrammer, hex_path)`, either inline (Step 3 above) or via
   `flash_husky.py` from a terminal.
4. Go back to `Lab 2_1B ... (MAIN).ipynb` with `VERSION = 'HARDWARE'` to run the SPA
   attack against your new password, or use its "Hands-On: Check Your Own Guess" section
   to test guesses interactively against the board you just reflashed.

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2020. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>